# M2P3 Trimming and Quality Control

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

### Objectives
By the end of this notebook, you should be able to:
- Use **FastQC** to evaluate read quality.
- Use **Trimmomatic** to trim low-quality bases and remove adapters.

  
*[Workflow figure omitted from the public version (third-party source not confirmed).]*


### Tools
- `FastQC`: A read-quality assessment tool that generates summary reports and plots.
- `Trimmomatic`: A flexible read-trimming tool for Illumina NGS data.

## `FastQC` - Check Read Quality BEFORE and AFTER Trimming

`FastQC` is a QC tool designed to perform checks on the quality of high-throughput sequencing data. This tool, similar to `fastp`, helps you to evaluate necessary steps while pre-processing your sequencing data. 

When you run a `FastQC` command in the terminal, such as the one below:

`fastqc sample1.fastq.gz -o results/sample1/`

`FastQC` will do the following: 

1. Reads the `fastq` file.
 
   Recall from [M2L4](../lab/M2L4_Sequencing_File_Formats.ipynb), each read within a `fastq` file has entries like this:
   
    ```
    @READ_ID
    AGGCTTGGCTA...
    +
    IIIIHGFFFFF...
    ```
    
    
   The first line is the read ID, the second line is the sequence of the read, and the last line contains Phred quality scores encoded as ASCII
   characters, which tell you how confident the sequencer was in each base call.

2. Summarizes statistics about the individual reads within the `fastq` file, such as:
   - Read length
   - Total number of reads
   - GC content
   - Quality score distribution


3. Runs diagnostic modules to detect specific issues within the `fastq` file's reads:
   | Module                           | What It Measures                                         | Expected Output                                                                                                |
| -------------------------------- | -------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------- |
| **Basic Statistics**             | Number of reads, sequence length, GC content             | Conformation the file loaded correctly                                                                    |
| **Per Base Sequence Quality**    | Average quality score per base position across all reads | Should start high (≥30) and stay consistent. Drops at 3′ end can indicate sample degradation                                           |
| **Per Sequence Quality Scores**  | Average quality score per *read*                         | Should be skewed toward higher scores                                                                                  |
| **Per Base Sequence Content**    | %A, %T, %G, %C at each base position                     | Should be roughly even in random libraries. A large imbalance suggests bias (due to untrimmed adapters or low complexity) |
| **Per Base GC Content**          | GC % at each base position                               | Should match the expected genomic GC profile                                                                           |
| **Per Sequence GC Content**      | GC % across all reads                                    | Should form a single smooth peak (Multiple peaks can suggest contamination)                                            |
| **Per Base N Content**           | Fraction of bases that are “N” (ambiguous)               | Should be near 0%                                                                                                      |
| **Sequence Length Distribution** | Read lengths                                             | Should match your sequencing setup (e.g., 150 bp for Illumina PE150)                                                   |
| **Sequence Duplication Levels**  | % of duplicated reads                                    | High duplication can indicate PCR bias or low library diversity                                                        |
| **Overrepresented Sequences**    | Reads appearing much more than expected                  | Often adapter contamination or rRNA contamination                                                                      |
| **Adapter Content**              | Matches to known adapter sequences                       | Should be near 0. A high adapter content means trimming is needed                                                               |

5. Generates a report for analysis of the `fastq` file's quality
   - An `HTML` report is produced for easy reading.
   - Reports can be sent to programs such as `MultiQC` for analyzing multiple samples within a set or across treatments.   

## Use of `FastQC`

Follow along with this video and run the commands in the cell below within the terminal in a Hellbender Desktop Session ([OnDemand](https://hb-classes.missouri.edu)):


#SHIFT + ENTER
<iframe width="560" height="315" src="https://www.youtube.com/embed/jdOS_lZqlKc?si=jrBz6LFh8pOy6dc6" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
%%bash
#load modules
module load mamba
module load miniconda3

#make and env for fastqc and trimmomatic. From here on out, this course will generally use global environments; however, it's always good practice to create your own. 
mamba create -n trim_qc_env -c bioconda fastqc trimmomatic -y

#activate the trim_qc_env
source activate trim_qc_env

#change to your home directory
cd 

#Navigate to the symlink of your student folder from your home directory
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
cd "$DATA_DIR"/practice/m2p3   # created by: bash data/fetch_data.sh practice

#Make a directory for fastp results on the un-trimmed file
mkdir -p results/fastqc_raw

# Run FastQC on an example fastq file (These are paired-end reads)
fastqc data/ExampleFastQC100kReads_R1_001.fastq.gz data/ExampleFastQC100kReads_R2_001.fastq.gz -o results/fastqc_raw

#Now, let's evaluate these results! Navigate to the $DATA_DIR/practice/m2p3/results/fastqc_raw folder within your file browser and open each .html file. 


## Use of `Trimmomatic`

Trimming removes reads that will interfere with downstream analysis. These include reads containing adapters, removing low-quality bases at the start or ends of reads, and removing reads that are too short. When used on paired-end reads, `Trimmomatic` ensures mated reads stay matched together. 

In this example, we determined that the results are going to be identical because the example fastq files have already been trimmed and preprocessed, and we will move forward with trimming. Watch this video on how to run `Trimmomatic` within the same Hellbender Desktop session as before. We will also check the read quality after trimming. 


<iframe width="560" height="315" src="https://www.youtube.com/embed/TKxTCStd7AQ?si=Cb5sAHxCdZ3mejMJ" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
%%bash 
#If you closed the terminal, run the following commands: 
#load modules
module load mamba
module load miniconda3

#activate the trim_qc_env
source activate trim_qc_env

#change to your home directory
cd 

#Navigate to the symlink of your student folder from your home directory
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
cd "$DATA_DIR"/practice/m2p3/   # created by: bash data/fetch_data.sh practice

#****************************************IF YOU ARE FOLLOWING ALONG IN SUCCESION TO THE PREVIOUS FASTQC VIDEO, START HERE***********************
mkdir -p results/trimmed

#running trimmomatic on the paired-end reads
trimmomatic PE \
  data/ExampleFastQC100kReads_R1_001.fastq.gz \
  data/ExampleFastQC100kReads_R2_001.fastq.gz \
  results/trimmed/ExampleFastQC100kReads_R1_paired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R1_unpaired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R2_paired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R2_unpaired.fq.gz \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:20 MINLEN:50

mkdir -p results/fastqc_trimmed

#running fastqc on the trimmed reads
fastqc results/trimmed/*.fq.gz -o results/fastqc_trimmed

*The above  command:* 
```
trimmomatic PE \
  data/ExampleFastQC100kReads_R1_001.fastq.gz \
  data/ExampleFastQC100kReads_R2_001.fastq.gz \
  results/trimmed/ExampleFastQC100kReads_R1_paired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R1_unpaired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R2_paired.fq.gz \
  results/trimmed/ExampleFastQC100kReads_R2_unpaired.fq.gz \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:20 MINLEN:50
```


**NOTE: THERE IS NO ADAPTER REMOVAL WITHIN THIS COMMAND**

*Has the following parameters:*

| Parameter            | Description                                                    |
| -------------------- | -------------------------------------------------------------- |
| `PE`                 | Run in paired-end mode.                                        |
| `LEADING:3`          | Remove low-quality bases from the start of a read (Phred < 3). |
| `TRAILING:3`         | Remove low-quality bases from the end of a read (Phred < 3).   |
| `SLIDINGWINDOW:4:20` | Scan with a 4-base window; cut when the average quality < 20.  |
| `MINLEN:50`          | Discard reads shorter than 50 bp after trimming.               |


Example data set used for fastqc and trimmomatic available from: https://github.com/hartwigmedical/testdata/tree/master/100k_reads_hiseq